# Step 1: Setup prerequisites

In [ ]:
import os
import sys
from pymongo import MongoClient

# Add parent directory to path to import from utils
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))

In [ ]:
# If you are using your own MongoDB Atlas cluster, use the connection string for your cluster here
MONGODB_URI = os.environ.get("MONGODB_URI")
# Initialize a MongoDB Python client
mongodb_client = MongoClient(MONGODB_URI, appname="devrel-workshop-ai-agents")
# Check the connection to the server
mongodb_client.admin.command("ping")

### **Do not change the values assigned to the variables below**

In [ ]:
#  Database name
DB_NAME = "mongodb_genai_devday_agents"
# Name of the collection with full documents- used for summarization
FULL_COLLECTION_NAME = "mongodb_docs"
# Name of the collection for vector search- used for Q&A
VS_COLLECTION_NAME = "mongodb_docs_vs"
# Name of the vector search index
VS_INDEX_NAME = "mongodb_docs_vs_autoembed"

# Connect to the `VS_COLLECTION_NAME` collection.
vs_collection = mongodb_client[DB_NAME][VS_COLLECTION_NAME]
# Connect to the `FULL_COLLECTION_NAME` collection.
full_collection = mongodb_client[DB_NAME][FULL_COLLECTION_NAME]

# Step 2: Create agent tools

📚 https://docs.langchain.com/oss/python/langchain/tools#create-tools


In [ ]:
from langchain.agents import tool
import voyageai
from typing import List

### Vector Search

📚 https://www.mongodb.com/docs/vector-search/query/aggregation-stages/vector-search-stage/?deployment-type=atlas&embedding=auto&interface=driver&language=python#simple-query

In [ ]:
# Define a tool to retrieve relevant documents for a user query using vector search
@tool
def get_information_for_question_answering(user_query: str) -> str:
    """
    Retrieve information using vector search to answer a user query.

    Args:
    user_query (str): The user's query string.

    Returns:
    str: The retrieved information formatted as a string.
    """

    # Define an aggregation pipeline consisting of a $vectorSearch stage, followed by a $project stage
    # Set the number of candidates to 150 and only return the top 5 documents from the vector search
    # In the $project stage, exclude the `_id` field and include only the `body` field and `vectorSearchScore`
    # NOTE: Use the variable defined previously for the `index` field. Set the `path` field to `body`.
    pipeline = [
        {
            "$vectorSearch": {
                "index": VS_INDEX_NAME,
                "path": "body",
                "query": {
                    "text": user_query
                },
                "numCandidates": 150,
                "limit": 5,
            }
        },
        {
            "$project": {
                "_id": 0,
                "body": 1,
                "score": {"$meta": "vectorSearchScore"},
            }
        },
    ]

    # Execute the aggregation `pipeline` against the `vs_collection` collection and store the results in `results`
    results = vs_collection.aggregate(pipeline)
    # Concatenate the results into a string
    context = "\n\n".join([doc.get("body") for doc in results])
    return context

### Get page content

📚 https://www.mongodb.com/docs/manual/reference/method/db.collection.findOne/#return-all-but-the-excluded-fields

In [ ]:
# Define a tool to retrieve the content of a documentation page for summarization
@tool
def get_page_content_for_summarization(user_query: str) -> str:
    """
    Retrieve page content based on provided title.

    Args:
    user_query (str): The user's query string i.e. title of the documentation page.

    Returns:
    str: The content of the page.
    """
    # Query the documents where the `title` field is equal to the `user_query`
    query = {"title": user_query}
    # Only return the `body` field from the retrieved documents.
    # NOTE: Set fields to include to 1, those to exclude to 0. `_id` is included by default, so exclude that.
    projection = {"_id": 0, "body": 1}
    # Use the `query` and `projection` with the `find_one` method
    # to get the `body` of the document with `title` equal to the `user_query` from the `full_collection` collection
    document = full_collection.find_one(query, projection)
    if document:
        return document["body"]
    else:
        return "Document not found"

In [ ]:
# Create the list of tools
tools = [
    get_information_for_question_answering,
    get_page_content_for_summarization,
]

### Test out the tools


In [ ]:
# Test out the `get_information_for_question_answering` tool with the query "What are some best practices for data backups in MongoDB?"
# You should see a non-empty response
get_information_for_question_answering.invoke(
    "What are some best practices for data backups in MongoDB?"
)

In [ ]:
# Test out the `get_page_content_for_summarization` tool with page title "Create a MongoDB Deployment"
# You should see a non-empty response
get_page_content_for_summarization.invoke("Create a MongoDB Deployment")

# Step 3: Instantiate the LLM

In [ ]:
from langchain_core.load import load
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI

MAX_TOKENS = 4096

In [ ]:
# Obtain the Langchain LLM object using the `get_llm` function from the `utils`` module.
llm =  AzureChatOpenAI(
    azure_endpoint="https://gai-326.openai.azure.com/",
    azure_deployment="gpt-5.1",
    api_version="2024-12-01-preview",
    temperature=0,
    max_tokens=MAX_TOKENS,
)

In [ ]:
# Create a Chain-of-Thought (CoT) prompt template for the agent.
# This includes a system prompt with a placeholder for tool names, and a placeholder for messages i.e. user queries and assistant responses
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful AI assistant."
            " You are provided with tools to answer questions and summarize technical documentation related to MongoDB."
            " Think step-by-step and use these tools to get the information required to answer the user query."
            " Do not re-run tools unless absolutely necessary."
            " If you are not able to get enough information using the tools, reply with I DON'T KNOW."
            " You have access to the following tools: {tool_names}."
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

In [ ]:
# Fill in the prompt template with the tool names. This creates a new ChatPromptTemplate instance with the names of
# the tools pre-filled so that they don't need to be passed on each invocation of the agent at runtime.
# The messages placeholder will still need to be substituted on each runtime invocation of the agent.
prompt = prompt.partial(tool_names=", ".join([tool.name for tool in tools]))

<img src="skill-pill.png" alt="Skill Pill" width="75"/>

📚 https://docs.langchain.com/oss/python/langgraph/quickstart#1-define-tools-and-model

In [ ]:
# Bind the `tools` to the `llm` instantiated above. This makes the LLM model aware of the tools and their
# functionality, allowing it to request they be invoked as needed during a conversation.
bind_tools = llm.bind_tools(tools)

<img src="skill-pill.png" alt="Skill Pill" width="75"/>

📚 https://reference.langchain.com/python/langchain-core/runnables/base/Runnable/pipe (See Example)

In [ ]:
# Using the `|` operator, create a runnable sequence that passes the `prompt` output (the core prompt with
# the placeholders replaced by the partialled tool names and messages list), to the tool-augmented llm.
# This results in the system prompt, tool names, and messages being passed to the LLM on each invocation.
llm_with_tools = prompt | bind_tools

In [ ]:
# Test that the LLM is making the right tool calls
llm_with_tools.invoke(
    ["Give me a summary of the page titled Create a MongoDB Deployment."]
).tool_calls

In [ ]:
# Test that the LLM is making the right tool calls
llm_with_tools.invoke(
    ["What are some best practices for data backups in MongoDB?"]
).tool_calls

# Step 4: Define graph state

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

In [ ]:
# Define the graph state
# We are only tracking chat messages but you can track other attributes as well
# `add_messages` is a special helper function provided by LangGraph that specifies
# that any new messages should always be appended to the `messages` list rather than
# replacing the existing list.
class GraphState(TypedDict):
    messages: Annotated[list, add_messages]

# Step 5: Define graph nodes

In [ ]:
from langchain_core.messages import ToolMessage
from typing import Dict
from pprint import pprint

In [ ]:
# Define the agent node
def agent(state: GraphState) -> Dict[str, List]:
    """
    Agent node

    Args:
        state (GraphState): Graph state

    Returns:
        Dict[str, List]: Updates to messages
    """
    # Get the messages from the graph `state`
    messages = state["messages"]
    # Invoke `llm_with_tools` with `messages` using the `invoke` method
    # HINT: See Step 3 for how to invoke `llm_with_tools`
    result = llm_with_tools.invoke(messages)
    # Write `result` to the `messages` attribute of the graph state
    return {"messages": [result]}

In [ ]:
# Create a map of tool name to tool call
tools_by_name = {tool.name: tool for tool in tools}
pprint(tools_by_name)

In [ ]:
# Define tool node
def tool_node(state: GraphState) -> Dict[str, List]:
    """
    Tool node

    Args:
        state (GraphState): Graph state

    Returns:
        Dict[str, List]: Updates to messages
    """
    result = []
    # Get the list of tool calls from messages
    tool_calls = state["messages"][-1].tool_calls
    # A tool_call looks as follows:
    # {
    #     "name": "get_information_for_question_answering",
    #     "args": {"user_query": "What are Atlas Triggers"},
    #     "id": "call_H5TttXb423JfoulF1qVfPN3m",
    #     "type": "tool_call",
    # }
    # Iterate through `tool_calls`
    for tool_call in tool_calls:
        # Get the tool from `tools_by_name` using the `name` attribute of the `tool_call`
        tool = tools_by_name[tool_call["name"]]
        # Invoke the `tool` using the `args` attribute of the `tool_call`
        # HINT: See previous line to see how to extract attributes from `tool_call`
        observation = tool.invoke(tool_call["args"])
        # Append the result of executing the tool to the `result` list as a ToolMessage
        # The `content` of the message is `observation` i.e. result of the tool call
        # The `tool_call_id` can be obtained from the `tool_call`
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    # Write `result` to the `messages` attribute of the graph state
    return {"messages": result}

# Step 6: Define conditional edges

In [ ]:
from langgraph.graph import END

In [ ]:
# Define conditional routing function
def route_tools(state: GraphState):
    """
    Use in the conditional_edge to route to the tool node if the last message
    has tool calls. Otherwise, route to the end.
    """
    # Get messages from graph state
    messages = state.get("messages", [])
    if len(messages) > 0:
        # Get the last AI message from messages
        ai_message = messages[-1]
    else:
        raise ValueError(f"No messages found in input state to tool_edge: {state}")
    # Check if the last message has tool calls
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        # If yes, return "tools"
        return "tools"
    # If no, return END
    return END

# Step 7: Build the graph

In [ ]:
from langgraph.graph import StateGraph, START
from IPython.display import Image, display

In [ ]:
# Instantiate the graph
graph = StateGraph(GraphState)

📚 https://docs.langchain.com/oss/python/langgraph/graph-api#nodes

In [ ]:
# Add nodes to the `graph` using the `add_node` method
# Add a `agent` node. The `agent` node should run the `agent` function
graph.add_node("agent", agent)
# Add a `tools` node. The `tools` node should run the `tool_node` function
graph.add_node("tools", tool_node)

📚 https://docs.langchain.com/oss/python/langgraph/graph-api#normal-edges

In [ ]:
# Add fixed edges to the `graph` using the `add_edge` method
# Add an edge from the START node to the `agent` node
graph.add_edge(START, "agent")
# Add an edge from the `tools` node to the `agent` node
graph.add_edge("tools", "agent")

📚 https://docs.langchain.com/oss/python/langgraph/graph-api#conditional-edges

In [ ]:
# Use the `add_conditional_edges` method to add a conditional edge from the `agent` node to the `tools` node
# based on the output of the `route_tools` function
graph.add_conditional_edges(
    "agent",
    route_tools,
    {"tools": "tools", END: END},
)

In [ ]:
# Compile the `graph`
app = graph.compile()

In [ ]:
# Visualize the graph
app

# Step 8: Execute the graph

In [ ]:
# Define a function to execute the graph and stream outputs from each step
def execute_graph(user_input: str) -> None:
    """
    Stream outputs from the graph execution

    Args:
        user_input (str): User query string
    """
    # Stream outputs from each step in the graph
    for step in app.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        # Stream full value of the state after each step
        stream_mode="values",
    ):
        # Print the latest message from the step
        step["messages"][-1].pretty_print()

In [ ]:
# Test the graph execution to view end-to-end flow
execute_graph("What are some best practices for data backups in MongoDB?")

In [ ]:
# Test the graph execution to view end-to-end flow
execute_graph("Give me a summary of the page titled Create a MongoDB Deployment")

In [ ]:
# Ask a follow up question - what does the agent remember about previous interactions?
execute_graph("What did I just ask you?")

# Step 9: Add short-term memory to the agent

In [ ]:
from langgraph.checkpoint.mongodb import MongoDBSaver

In [ ]:
# Initialize a MongoDB checkpointer
checkpointer = MongoDBSaver(mongodb_client)

In [ ]:
# Instantiate the graph with the checkpointer
app = graph.compile(checkpointer=checkpointer)

📚 https://docs.langchain.com/oss/python/langgraph/persistence#threads

In [ ]:
# Update the graph execution function to handle thread IDs
def execute_graph(thread_id: str, user_input: str) -> None:
    """
    Stream outputs from the graph execution

    Args:
        thread_id (str): Thread ID for the checkpointer
        user_input (str): User query string
    """
    # Create a runtime config for the thread ID `thread_id`
    config = {"configurable": {"thread_id": thread_id}}
    # Stream outputs from each step in the graph
    for step in app.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        # Pass the config as an additional parameter
        config,
        stream_mode="values",
    ):
        # Print the latest message from the step
        step["messages"][-1].pretty_print()

In [ ]:
# Test graph execution with thread ID
execute_graph(
    "1",
    "What are some best practices for data backups in MongoDB?",
)

In [ ]:
# Follow-up question to ensure message history works
execute_graph(
    "1",
    "What did I just ask you?",
)

# 🦹‍♀️ Step 10: Add long-term memory to the agent

📚 https://docs.langchain.com/oss/python/langgraph/add-memory#use-semantic-search

In [ ]:
from langgraph.store.mongodb import MongoDBStore, create_vector_index_config
from langchain_voyageai import VoyageAIEmbeddings
import uuid

In [ ]:
# Initialize MongoDB collection for long-term memory storage
memory_collection = mongodb_client[DB_NAME]["memories"]

In [ ]:
# Initialize the MongoDB long-term memory store with Voyage embeddings to retrieve memories using vector search
mongodb_store = MongoDBStore(
    collection=memory_collection,
    index_config=create_vector_index_config(
        embed=VoyageAIEmbeddings(model="voyage-4"),
        dims=1024,
    ),
    auto_index_timeout=60,
)

In [ ]:
# Create a tool to save memories to the MongoDB store
@tool
def save_memory(memory: str) -> str:
    """
    Save important facts and preferences about the user for future conversations.

    Args:
    memory: The information to remember
    """
    mongodb_store.put(
        # Namespace for the memory entry. You can also have sub-namespaces to store different types of memories, eg: ("user_1", "preferences")
        # Has to be a tuple, even if it contains empty values
        ("user_1",),
        # Unique memory ID
        key=str(uuid.uuid4()),
        # Content of the memory. Needs to be a dictionary.
        value={"text": memory},
    )
    return f"Memory saved: {memory}"

In [ ]:
# Update the tools list to include the `save_memory` tool
tools = [
    get_information_for_question_answering,
    get_page_content_for_summarization,
    save_memory,
]
tools_by_name = {tool.name: tool for tool in tools}
# Bind tools to the LLM
bind_tools = llm.bind_tools(tools)

In [ ]:
# Update the agent node to retrieve relevant long-term memories when responding
def agent(state: GraphState) -> Dict[str, List]:
    """
    Agent node

    Args:
        state (GraphState): Graph state

    Returns:
        Dict[str, List]: Updates to messages
    """
    # Get `messages` from the graph `state`
    messages = state["messages"]
    # Search for relevant long-term memories using the user's last message
    memories = mongodb_store.search(("user_1",), query=messages[-1].content, limit=10)
    # Format retrieved memories into a string.
    memories = "\n".join(m.value["text"] for m in memories)
    memories = memories if memories else "No memories stored yet for this user."
    # Build system prompt too include memories
    system_prompt = (
        "You are a helpful AI assistant."
        "You are provided with tools to answer questions and summarize technical documentation related to MongoDB."
        "Think step-by-step and use these tools to get the information required to answer the user query."
        "Do not re-run tools unless absolutely necessary."
        "If you are not able to get enough information using the tools, reply with I DON'T KNOW."
        f" You have access to the following tools: {', '.join([t.name for t in tools])}."
        "If the user shares any preferences, extract them and save them as memories using the save_memory tool."
        "Use past user preferences to personalize future conversations."
        "Past user memories:"
        f"{memories}"
    )
    # Invoke the tool-augmented LLM with the system prompt and messages as input
    result = bind_tools.invoke(
        [
            {"role": "system", "content": system_prompt},
            *messages,
        ]
    )
    # Write the `result` to the `messages` attribute of the graph state
    return {"messages": [result]}

In [ ]:
# Rebuild the agent graph
graph = StateGraph(GraphState)
graph.add_node("agent", agent)
graph.add_node("tools", tool_node)
graph.add_edge(START, "agent")
graph.add_edge("tools", "agent")
graph.add_conditional_edges(
    "agent",
    route_tools,
    {"tools": "tools", END: END},
)

In [ ]:
# Compile the graph with the MongoDB checkpointer for short-term memory as well as the MongoDB memory store for long-term memory
app = graph.compile(checkpointer=checkpointer, store=mongodb_store)

In [ ]:
# Test creating memories in thread 1 for user user_1
execute_graph(
    "1",
    "Remember that I prefer detailed explanations with code examples when learning about MongoDB features.",
)

In [ ]:
# Ask a follow-up question in the same thread- ensures short-term memory is working
execute_graph("1", "What do you know about my learning preferences?")

In [ ]:
# As a follow-up question in a NEW thread- ensures long-term, cross-session memory is working
execute_graph(
    "2", "Hi! Do you remember anything about how I like to learn about MongoDB?"
)

In [ ]:
# Ask about MongoDB Search - the agent should provide detailed explanations with examples based on saved preferences
execute_graph("2", "What is MongoDB Search?")